# Part 3 — Agent Skills: teaching an agent how, not just what

Task: Run the full BRCA subtype-prediction pipeline on the
fitted MOFA model and give a one-paragraph interpretation, then generate the
diagnostic plots and say which one best supports the claim that MOFA captures
subtype biology.

Same `src/mofa_tools.py` backend as Part 1 and Part 2, but here a coding agent
(Pi) reaches it by running Python through its own `bash` tool, guided by a skill.

## Learning objectives

By the end of this notebook you should be able to:
- Explain the shift from a fixed list of tools (Parts 1-2) to a coding agent with general-purpose tools (`bash`, file read/write).
- Explain what an Agent Skill is, and what problem it solves that tools and MCP do not.
- Run the same task through Pi with and without a skill loaded, and identify which differences are new capability versus behaviour.
- Explain why "load the cached model, never refit" belongs in a skill rather than a tool signature.

## 0. From tool-calling to coding agents, why the paradigm shifted

Where we've been. In nb01 we handed the model a fixed list of typed
functions (`@tool`); in nb02 we served the same functions over MCP. Both share a
mental model: a brain wired to a hand-curated set of tools.

**The shift.** Modern agent design moved toward giving the model a few
general tools: read a file, write a file, run a shell command, inside a
real repository/filesystem. Once an agent can run `bash` and edit files, it
can install a package, run a script, inspect data, call a CLI, with no
pre-built tool for each. The interface becomes the filesystem + shell. This
is the coding-agent paradigm: Claude Code, Codex, Cursor, OpenHands, Pi.

Why now? Models got good enough at long-horizon reasoning and code; a few
primitives (`bash` + file I/O) generalize better than 50 brittle bespoke tools;
and the repo becomes shared, persistent memory across steps.

## 1. The new problem: how do you give such an agent know-how?

A coding agent with `bash` + files can do almost anything, but it doesn't
know your specifics: your procedure for interpreting MOFA factors, that it
must load the cached model rather than re-fit, your house answer format, the
biomedical-caution rules. Cramming all that into the system prompt costs tokens
every turn and becomes a junk drawer.

### Agent Skills & `SKILL.md`

Agent Skills package procedural knowledge into a portable, version-controlled
folder loaded on demand. A skill is a directory with a `SKILL.md` (YAML
frontmatter `name`+`description`, then instructions), optionally `scripts/`,
`references/`, `assets/`.

Progressive disclosure makes it scale: three stages, cheapest first:

1. Discovery: the agent sees only each skill's `name` + `description`.
2. Activation: on a matching task it reads the full `SKILL.md`.
3. Execution: it follows the instructions, running bundled scripts only if needed.

So dozens of skills cost almost nothing idle. Agent Skills were released by
Anthropic as an [open standard](https://agentskills.io/specification) and adopted
across Claude Code, Cursor, Codex, Pi, and others.

### The harness

The model can't itself read files or run `bash`, it only emits text/tool-call
requests. The harness (agent runtime) implements those primitives, runs the
loop, manages context, and discovers/loads skills. Claude Code, Codex, and Pi
are harnesses. Model : harness :: engine : car. Skills are a format the harness
loads, not part of the model.

## 2. How skills overlap with tools and MCP

| | What it adds | Form | Answers… |
|---|---|---|---|
| Tools (nb01) | a new capability | a typed function the model calls | what can the agent do? |
| MCP (nb02) | the same capabilities, portably | a standard server for tools/resources/prompts | where do they live, who reuses them? |
| Skills (nb03) | know-how / workflow | a `SKILL.md` folder of instructions | how should it do it, and when? |

> Tools & MCP = what the agent can do. Skills = what it should do, and how.

The edges blur honestly: a skill can bundle `scripts/` (functionally tools); a
skill can tell the agent to use an MCP server; an MCP server also serves
prompts (templated instructions) which overlap with skills. They compose,
tools/MCP supply capabilities; skills supply the procedural knowledge that
decides which to use, in what order, and how to report, loaded only when
relevant.

## 3. For this notebook: Pi

We drive the skill with **Pi**, a minimal terminal coding harness with
first-class skills support ([pi.dev](https://pi.dev)). Install separately:

```bash
npm install -g --ignore-scripts @earendil-works/pi-coding-agent
# or: curl -fsSL https://pi.dev/install.sh | sh
```

> Different shape. Unlike nb01/nb02 (pure `import` + an in-process loop),
> Pi is a terminal harness, so we drive it via `subprocess`. The same
> `skills/mofa-multiomics-agent.SKILL.md` works with any skills-compatible
> harness (Claude Code, Goose, …), the skill file doesn't change.

## 4. Setup and the skill we'll hand to Pi

Bring-your-own-key from `.env`. We locate the Pi CLI and make Pi's `bash` tool
use this **`eccb`** interpreter, so it can `import src.mofa_tools` and load the
cached MOFA model.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not found (.env in project root)."

SKILL_PATH = PROJECT_ROOT / "skills" / "mofa-multiomics-agent.SKILL.md"
MODEL = "anthropic/claude-haiku-4-5"

PI = shutil.which("pi") or str(Path.home() / ".local" / "bin" / "pi")
assert Path(PI).exists(), f"Pi CLI not found at {PI}. Install it (see https://pi.dev)."

# Pi answers by running Python via its bash tool, so its shell must use the eccb
# interpreter where src.mofa_tools + mofax import.
ENV_BIN = str(Path(sys.executable).parent)
PI_ENV = {**os.environ, "PATH": f"{ENV_BIN}:{Path(PI).parent}:{os.environ.get('PATH', '')}"}
print("python    :", sys.executable)

The skill is a valid Agent Skill: YAML frontmatter then markdown. It contains
no model-specific code, just procedural knowledge: when to use it, the
"load the cached model, never re-fit" rule, evidence discipline, the fixed answer
format, biomedical caution, and a verified "how to query this repo" recipe
pointing at `src/mofa_tools.py`.

In [ ]:
print(SKILL_PATH.read_text())

## 5. Run Pi as a real harness, with the skill

We hand a prompt to a coding agent and let it run the loop, choose its
`read`/`bash`/`edit`/`write` tools, and load the skill. Command shape:

```bash
pi -p "<task>" --skill skills/mofa-multiomics-agent.SKILL.md \
   --model anthropic/claude-haiku-4-5 --approve
```

- `-p` runs headless; `--skill` loads our skill; `--approve` trusts project-local
  files (Pi runs in `PROJECT_ROOT`, so `bash` can run `python -c "from
  src.mofa_tools import ..."`).

Watch the output follow the skill's Answer / Evidence Used / Interpretation /
Limitations structure, that comes from the skill, not from us.

In [ ]:
def run_pi(prompt: str, with_skill: bool, model: str = MODEL, timeout: int = 600):
    """Invoke the Pi coding agent headlessly and return its final text answer."""
    cmd = [PI, "-p", prompt, "--model", model, "--approve"]
    cmd += ["--skill", str(SKILL_PATH)] if with_skill else ["--no-skills"]
    proc = subprocess.run(cmd, cwd=PROJECT_ROOT, env=PI_ENV,
                          capture_output=True, text=True, timeout=timeout)
    if proc.returncode != 0:
        print("[pi stderr]\n", proc.stderr[-2000:])
    return proc.stdout.strip()

### Query 9: run the pipeline + interpret (spends API tokens)

In [ ]:
PIPELINE_TASK = (
    "Run the full breast-cancer subtype-prediction pipeline on the fitted MOFA "
    "model (load the cached model, do not re-fit): report which factors are "
    "active, which factor is most associated with PAM50 subtype, and the held-out "
    "classification performance. Then give a one-paragraph interpretation.")
print(run_pi(PIPELINE_TASK, with_skill=True))

### Query 10: generate diagnostics + justify  (spends API tokens)

In [ ]:
#################################################
# Write a task asking Pi to generate the standard MOFA diagnostic plots into
# outputs/, and say which single plot best supports the claim that MOFA captures
# breast-cancer subtype biology, and why.
PLOTS_TASK = # YOUR PROMPT HERE
print(run_pi(PLOTS_TASK, with_skill=True))
#################################################

## 6. Run without the skill, compare behaviour (spends API tokens)

Same agent, model, task, but `--no-skills`. Pi can still find
`src/mofa_tools.py` by exploring the repo, so the capability is unchanged.
What changes is behaviour: without the skill it has no instruction to load
the cached model (it might try to re-fit), to ground every claim in tool output,
to use the fixed answer format, or to add biomedical caveats.

In [ ]:
print(run_pi(PIPELINE_TASK, with_skill=False))

## Reflection

- Same backend, three interfaces. nb01 bound the functions as tools, nb02
  served them over MCP, here Pi ran `python` through `bash`, same MOFA results.
  Where does "capability" actually live?
- What did the skill change? Diff the §5 (with-skill) and §6 (no-skill)
  answers: which differences are format, which evidence discipline, which
  the "don't re-fit" rule? None are new capabilities.
- The re-fit trap. Why is "load the cached model, never fit" exactly the kind
  of knowledge that belongs in a skill rather than a tool signature?
- Progressive disclosure: we passed `--skill` explicitly. Why does
  discovery-by-description matter once you have 50 skills?
- Overlap: the skill told the agent how to call `src/mofa_tools.py`. If those
  were exposed via the nb02 MCP server instead, what in the skill's "how to
  query" section changes, and what stays the same?